In [7]:
!pip install pandas requests python-dotenv tqdm


You should consider upgrading via the '/Users/moriseiitsu/Dropbox/IDEAMAPS/ideamaps-models/models/emergency-maternal-care/scripts/.venv_requirements_test/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import os, time, json
import pandas as pd
import requests
from dotenv import load_dotenv
from tqdm import tqdm

/Users/moriseiitsu/Dropbox/IDEAMAPS/ideamaps-models/models/emergency-maternal-care/scripts/.venv_requirements_test/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")


In [3]:
INPUT_CSV = "../data-inputs/hc-facilities-pasto-repsminsalud_fixed.csv"                 
OUTPUT_CSV = "../data-inputs/hc-facilities-pasto-repsminsalud_geocoded.csv"               
CACHE_FILE = "../data-inputs/hc-facilities-pasto-repsminsalud_geocode_cache.json"           

ADDRESS_COL = "address_geocoding"                     
COUNTRY_COMPONENT = "country:CO"            

URL = "https://maps.googleapis.com/maps/api/geocode/json"

In [ ]:
def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_cache(cache: dict):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def geocode_one(address: str, session: requests.Session, max_retries: int = 5) -> dict:
    params = {"address": address, "key": API_KEY}
    if COUNTRY_COMPONENT:
        params["components"] = COUNTRY_COMPONENT

    backoff = 1.0
    for _ in range(max_retries):
        r = session.get(URL, params=params, timeout=25)
        data = r.json()
        status = data.get("status")

        if status == "OK" and data.get("results"):
            top = data["results"][0]
            loc = top["geometry"]["location"]
            return {
                "status": "OK",
                "latitude": loc.get("lat"),
                "longitude": loc.get("lng"),
                "formatted_address": top.get("formatted_address"),
                "place_id": top.get("place_id"),
                "error": "",
            }

        if status == "OVER_QUERY_LIMIT":
            time.sleep(backoff)
            backoff *= 2
            continue

        return {
            "status": status,
            "latitude": None,
            "longitude": None,
            "formatted_address": None,
            "place_id": None,
            "error": data.get("error_message", ""),
        }

    return {
        "status": "OVER_QUERY_LIMIT",
        "latitude": None,
        "longitude": None,
        "formatted_address": None,
        "place_id": None,
        "error": "retry exhausted",
    }


def main():
    df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)

    if ADDRESS_COL not in df.columns:
        raise SystemExit(f"Could not find column '{ADDRESS_COL}'. Columns are: {list(df.columns)}")

    # Add output columns (if missing)
    for c in ["geocode_status", "latitude", "longitude", "formatted_address", "place_id", "geocode_error"]:
        if c not in df.columns:
            df[c] = ""

    cache = load_cache()

    with requests.Session() as session:
        for i, addr in tqdm(df[ADDRESS_COL].items(), total=len(df)):
            address = str(addr).strip()

            if not address:
                df.at[i, "geocode_status"] = "INVALID_REQUEST"
                df.at[i, "geocode_error"] = "empty address"
                continue

            cache_key = f"{COUNTRY_COMPONENT or ''}|{address}"
            if cache_key in cache:
                result = cache[cache_key]
            else:
                result = geocode_one(address, session)
                cache[cache_key] = result
                save_cache(cache)

            df.at[i, "geocode_status"] = result["status"]
            df.at[i, "latitude"] = "" if result["latitude"] is None else result["latitude"]
            df.at[i, "longitude"] = "" if result["longitude"] is None else result["longitude"]
            df.at[i, "formatted_address"] = result["formatted_address"] or ""
            df.at[i, "place_id"] = result["place_id"] or ""
            df.at[i, "geocode_error"] = result["error"] or ""

            time.sleep(0.05)  # gentle rate limiting

    # ONLY want address + coords, use this instead:
    # df[[ADDRESS_COL, "latitude", "longitude"]].to_csv(OUTPUT_CSV, index=False)

    df.to_csv(OUTPUT_CSV, index=False)
    print("DONE:", OUTPUT_CSV)


if __name__ == "__main__":
    main()

In [6]:
in_file = "../data-inputs/hc-facilities-pereiradosquebradas_geocoded.csv"
out_file = in_file

df = pd.read_csv(in_file)

# 1) remove columns
cols_to_drop = ["ownership", "primaryhc-yn", "basic-emoc-y", "comprehensive-emoc-y", "place_id", "geocode_error", "formatted_address", "geocode_error"]
df = df.drop(columns=cols_to_drop, errors="ignore")

# 2) rename columns
rename_map = {
    "lat": "latitude",
    "lng": "longitude",
    "geocode_status": "status",
    # "address_geocoding": "address",
}
df = df.rename(columns=rename_map)

df.to_csv(out_file, index=False)
print("Saved:", out_file)


Saved: ../data-inputs/hc-facilities-pereiradosquebradas_geocoded.csv
